# Magnetic braking

Compare ideal torsional evolution at three seed field strengths, with and without Maxwell feedback. The reference crossing time is $t_A=(r_o-r_i)/B_{p,\mathrm{rms}}$.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
from nsinduct.shell import (
    run_case,
)

base = dict(
    ntheta=16,
    nr=24,
    r_inner=0.3,
    r_outer=1.0,
    field_strength=0.07,
    eta=0.0,
    nu=0.0,
    timestep=0.01,
    output_dt=0.1,
    snapshot_count=3,
    stop_time=12.0,
    lorentz_feedback=True,
)

In [ ]:
fields = [0.035, 0.07, 0.14]
runs = [
    run_case(base | dict(field_strength=b, stop_time=25.0, name=f"B{b}"))
    for b in fields
]
control = run_case(base | dict(stop_time=25.0, lorentz_feedback=False, name="control"))
print(" B_rms    t_A      first sampled half-energy time   t_half/t_A")
for b, run in zip(fields, runs):
    th = run["summary"]["braking_time_half_Ekin"]
    print(b, 0.7 / b, th, float(th) / (0.7 / b) if th != "not_reached" else None)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
for b, run in zip(fields, runs):
    rr = run["rows"]
    tt = np.array([r["time"] for r in rr])
    ax[0].plot(tt * b / 0.7, [r["E_kin"] / rr[0]["E_kin"] for r in rr], label=str(b))
rr = runs[1]["rows"]
tt = [r["time"] for r in rr]
ax[1].plot(tt, [r["E_kin"] for r in rr], label="kinetic, feedback")
ax[1].plot(tt, [r["E_tor"] for r in rr], label="toroidal, feedback")
ax[1].plot(
    [r["time"] for r in control["rows"]],
    [r["E_tor"] for r in control["rows"]],
    "--",
    label="toroidal, prescribed flow",
)
ax[0].set(xlabel=r"$t/t_A$", ylabel=r"$E_{kin}/E_{kin}(0)$")
ax[0].legend(title="Bp RMS")
ax[1].set(xlabel=r"$t$", ylabel="energy")
ax[1].set_yscale("symlog", linthresh=0.1)
ax[1].legend(fontsize=8)
plt.show()
rr = runs[1]["rows"]
print(
    "Maximum normalized cumulative budget residual:",
    max(abs(r["cumulative_energy_residual"]) for r in rr),
)
print("No-feedback final shear fraction:", control["summary"]["final_shear_fraction"])
assert control["summary"]["final_shear_fraction"] == 1.0

The curves nearly coincide when plotted against the scaled time. Without feedback the prescribed flow supplies the toroidal magnetic energy. Ideal feedback permits reversible energy exchange.

## Matched initial conditions

Compare the backends using the same continuous field and shear normalization. Each curve uses its own initial kinetic energy as the energy unit.

In [ ]:
import logging

logging.getLogger("dedalus").setLevel(logging.WARNING)

from nsinduct.dedalus_backend import run_case as run_spectral, normalization
from nsinduct.shell import make_grid, rk4_step, energies

cfg = dict(ntheta=16, nr=24, field_strength=0.07, eta=2e-5, nu=0.0,
           lorentz_feedback=True, stop_time=1.0, timestep=0.005,
           output_dt=0.05, snapshot_count=3)
spectral_feedback = run_spectral(cfg)
norm = normalization(cfg)
g_fd = make_grid(dict(ntheta=24, nr=36))
r, theta = g_fd.radius2d, g_fd.theta2d
h = ((r - 0.3)*(1 - r))**2
hp = 2*(r - 0.3)*(1 - r)*(1.3 - 2*r)
br = 2*norm["poloidal_scale"]*h*np.cos(theta)/r**2
bt = -norm["poloidal_scale"]*hp*np.sin(theta)/r
raw = (1 + 0.18*np.cos(2*np.pi*(r - 0.3)/0.7))*(3*np.cos(theta)**2 - 1)/2
omega_fd = (raw - norm["omega_mean"])/norm["omega_amplitude"]
q_fd = np.zeros_like(omega_fd)
fd_energy = [energies(g_fd, omega_fd, q_fd, br, bt)]
for _ in range(200):
    omega_fd, q_fd = rk4_step(g_fd, omega_fd, q_fd, br, bt, 0.005, 2e-5, 0, True)
    fd_energy.append(energies(g_fd, omega_fd, q_fd, br, bt))
fd_time = np.linspace(0, 1, len(fd_energy))
rows = spectral_feedback["rows"]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
for ax, key, fd_key in zip(axes, ["E_kin", "E_tor"], ["kinetic", "magnetic_toroidal"]):
    ax.plot([row["time"] for row in rows], [row[key]/rows[0]["E_kin"] for row in rows], label="Dedalus 16×24")
    ax.plot(fd_time, [e[fd_key]/fd_energy[0]["kinetic"] for e in fd_energy], "--", label="FD 24×36")
    ax.set(xlabel=r"$t$", ylabel=key + " / initial kinetic energy")
    ax.legend()
plt.show()
print("Dedalus normalized budget residual:", spectral_feedback["summary"]["max_cumulative_energy_residual"])
print("Dedalus angular-momentum change:", rows[-1]["angular_momentum_total"] - rows[0]["angular_momentum_total"])
